## Part 2: Stimuli

Now what we want to do is plot the predicted v the observed and the correlation.

We will do all preferred stimuli and then non-preferred stimuli in the same plot

In [6]:
%load_ext autoreload
%autoreload 2

# standard imports
import os
import numpy as np
from sklearn.linear_model import Ridge
import torch
import re
from sklearn.model_selection import StratifiedKFold
import scipy
from tqdm.notebook import tqdm
import pandas as pd
import warnings
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from joblib import dump, load

# custom imports

from compcats.models import resnet50_openaiclip
from compcats.datasets.fmri import murty185
from compcats.utils import model_utils, misc_utils, responsecomparison_utils
from compcats.encoder import encode

MODELS_DIR = '/Users/adipani3/projects/catsel/comparing-category-selectivity/preprint/models'
os.makedirs(MODELS_DIR, exist_ok=True)
MURTY185_DIR = '/Users/adipani3/datasets/murty185'
FMRI_DIR = f'{MURTY185_DIR}/DPNT'
RESULTS_DIR = 'results'
EMS_DIR = 'trained_ems'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(EMS_DIR, exist_ok=True)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=ImportWarning)
warnings.filterwarnings("ignore", category=PendingDeprecationWarning)
warnings.filterwarnings("ignore", category=Warning)  # catch-all for any base warnings
warnings.filterwarnings("ignore")

In [8]:
model = resnet50_openaiclip.get_model()
layers = model.relevant_layer_names

In [9]:
pattern = r"layer[1-4]\.\d+\.conv\d+"

filtered_layers = []
for l in layers:
    if re.search(pattern,l):
        filtered_layers.append(l)

# at this point I only have the layers I care about

nr_layers = len(filtered_layers)

In [10]:
# to answer earlier question, yes I need a dataset class
# one more question about the save_dir - can I just create a folder that is pretrained_models/name of model? or what is the best way
# X = model.generate_dataset_activations(layers,dataset,save_dir,num_workers=4)
X = model_utils.get_dataset_activations(
    MODELS_DIR, MURTY185_DIR, model, filtered_layers, 'murty185', device='cpu'
)
for key, value in X.items():
    X[key] = value.flatten(start_dim=1)

In [11]:
categories = murty185.LABELS
categories = np.array(categories)

In [12]:
all_files = os.listdir(FMRI_DIR)
Y_all_updated = {} # Y_all['group']['nr as int']['froi']
frois_considered = ['lffa', 'rffa', 'leba', 'reba']
for f in all_files:
    subid, froi, _ = f.split('.')[0].split('_')
    if froi not in frois_considered:
        continue
    if froi not in Y_all_updated:
        Y_all_updated[froi] = {}
    subtype = 'dp' if 'dp' in subid else 'nt'
    Y_all_updated[froi][subid] = {
        'data': torch.load(f'{FMRI_DIR}/{f}').cpu().numpy(),
        'subtype': subtype
    }
for froi, subdict in Y_all_updated.items():
    Y_all_updated[froi] = dict(sorted(subdict.items()))

udpated_subject_numbers_dict = {} # subject type -> [subject number]
for froi, subdict in Y_all_updated.items():
    udpated_subject_numbers_dict[froi] = list(subdict.keys())

In [13]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42) # 5fold; default seed is 42

In [14]:
prior_results_csv_name = 'ccn_pt1.csv'

df_prior_results = pd.read_csv(f'{RESULTS_DIR}/{prior_results_csv_name}')

In [18]:
df_prior_results

,subject_type,froi,test_subject,model,layer,mean_pearsonr,mean_rdmr
0,nt,lffa,1,resnet50openaiclip,layer4.2.conv1,0.738986,0.457658
1,nt,lffa,2,resnet50openaiclip,layer4.2.conv1,0.867785,0.654342
2,nt,lffa,3,resnet50openaiclip,layer4.2.conv1,0.856675,0.586247
3,nt,lffa,4,resnet50openaiclip,layer4.2.conv1,0.865600,0.651289
4,nt,lffa,5,resnet50openaiclip,layer4.2.conv1,0.825356,0.610292
...,...,...,...,...,...,...,...
1915,dp,reba,6,resnet50openaiclip,layer2.2.conv2,0.605192,0.288243
1916,dp,reba,6,resnet50openaiclip,layer2.2.conv3,0.598204,0.282202
1917,dp,reba,6,resnet50openaiclip,layer2.3.conv1,0.614369,0.307998
1918,dp,reba,6,resnet50openaiclip,layer2.3.conv2,0.608211,0.273335


In [ ]:
# Decide the best layer per froi
best_layer_df = (
    df_prior_results
    .groupby(['froi', 'layer'], as_index=False)['mean_pearsonr']
    .mean()
    .rename(columns={'mean_pearsonr': 'mean_pearsonr_across_subjects'})
    .sort_values(['froi', 'mean_pearsonr_across_subjects'], ascending=[True, False])
    .drop_duplicates(subset=['froi'])
    .reset_index(drop=True)
)
display(best_layer_df)

,froi,layer,mean_pearsonr_across_subjects
0,leba,layer4.1.conv1,0.710119
1,lffa,layer4.1.conv1,0.770900
2,reba,layer4.1.conv1,0.780866
3,rffa,layer4.1.conv1,0.784870


In [23]:
csv_name = 'ccn_stimuli_responses_acrossgroup.csv'
csv_path = f'{RESULTS_DIR}/{csv_name}'

if os.path.exists(csv_path):
    csv_loaded = pd.read_csv(csv_path)
    csv_rows_list = csv_loaded.to_dict('records')
else:
    csv_rows_list = []

frois_considered = ['lffa', 'rffa', 'leba', 'reba']

for froi in frois_considered:
    print(f'Processing roi: {froi}')

    y_subtypes = np.array([v['subtype'] for v in Y_all_updated[froi].values()])
    y_array = np.stack([v['data'] for v in Y_all_updated[froi].values()])
    subject_ids = np.array(udpated_subject_numbers_dict[froi])

    # Use one layer per ROI (already computed above)
    layer_row = best_layer_df[best_layer_df['froi'] == froi]
    if len(layer_row) == 0:
        continue
    layer_name = layer_row.iloc[0]['layer']

    X_layer = X[layer_name]
    split_indices = list(skf.split(X_layer, categories))

    # Train exactly one mapper per (train subject, split), then test on all subjects
    for tridx in np.arange(y_array.shape[0]):
        train_subj_id = subject_ids[tridx]
        train_subj_group = y_subtypes[tridx]
        train_subj_all_responses = y_array[tridx, :, :]

        print(f'  Training subject: {train_subj_id}')

        for spidx, (train_index, test_index) in enumerate(split_indices):
            X_train = X_layer[train_index].cpu().numpy()
            X_test = X_layer[test_index].cpu().numpy()

            # Fit once for this train subject + split
            train_responses = train_subj_all_responses[train_index, :]
            em_mapper = Ridge(alpha=0.01)
            em_mapper.fit(X_train, train_responses)

            # Save one mapper per train subject + split + layer
            save_em_path = (
                f"{EMS_DIR}/{froi}_{train_subj_group + train_subj_id[-1]}_"
                f"{spidx}_{layer_name}.joblib"
            )
            if not os.path.exists(save_em_path):
                dump(em_mapper, save_em_path)

            # Shared predictions for this split (independent of test subject)
            y_pred = em_mapper.predict(X_test)
            y_pred_mean = y_pred.mean(axis=-1)

            for tidx in np.arange(y_array.shape[0]):
                test_subj_id = subject_ids[tidx]
                test_subj_group = y_subtypes[tidx]
                test_subj_data = y_array[tidx, :, :]
                test_responses = test_subj_data[test_index, :]
                test_responses_mean = test_responses.mean(axis=-1)

                row_params = {
                    'froi': froi,
                    'test_subject_type': test_subj_group,
                    'test_subject': int(test_subj_id[-1]),
                    'train_subject_type': train_subj_group,
                    'train_subject': int(train_subj_id[-1]),
                    'model': model.name,
                    'layer': layer_name,
                    'split_index': spidx
                }

                if misc_utils.check_row_present(row_params, csv_rows_list):
                    continue

                test_pearsonr = scipy.stats.pearsonr(y_pred_mean, test_responses_mean)[0].item()
                test_rdmr = responsecomparison_utils.rdmr(
                    torch.tensor(test_responses),
                    torch.tensor(y_pred)
                )

                rows_to_add = []
                categories_test_fold = categories[test_index]
                for ciidx, current_img in enumerate(test_index):
                    rows_to_add.append({
                        **row_params,
                        'img_cat': categories_test_fold[ciidx],
                        'overall_img_idx': current_img,
                        'y_prediction': y_pred_mean[ciidx],
                        'y_observed': test_responses_mean[ciidx],
                        'test_pearsonr': test_pearsonr,
                        'test_rdmr': test_rdmr
                    })

                csv_rows_list.extend(rows_to_add)

    # Write once per ROI to reduce I/O overhead
    pd.DataFrame(csv_rows_list).to_csv(csv_path, index=False)

print('Done.')

Processing roi: lffa
  Training subject: sub-dp01
  Training subject: sub-dp03
  Training subject: sub-dp06
  Training subject: sub-nt01
  Training subject: sub-nt02
  Training subject: sub-nt03
  Training subject: sub-nt04
  Training subject: sub-nt05
  Training subject: sub-nt06
  Training subject: sub-nt07
Processing roi: rffa
  Training subject: sub-dp01
  Training subject: sub-dp03
  Training subject: sub-dp06
  Training subject: sub-nt01
  Training subject: sub-nt02
  Training subject: sub-nt03
  Training subject: sub-nt04
  Training subject: sub-nt05
  Training subject: sub-nt06
  Training subject: sub-nt07
Processing roi: leba
  Training subject: sub-dp01
  Training subject: sub-dp03
  Training subject: sub-dp06
  Training subject: sub-nt01
  Training subject: sub-nt02
  Training subject: sub-nt03
  Training subject: sub-nt04
  Training subject: sub-nt05
  Training subject: sub-nt06
  Training subject: sub-nt07
Processing roi: reba
  Training subject: sub-dp01
  Training subjec